In [15]:
# ============================================
# 🧠 Mini ChatGPT from Scratch (Pure PyTorch)
# No transformers, no pretrained weights
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


In [74]:
# # Create a simple Q&A dataset
# questions = [
#     "what is ai", "how are you", "what is python", "tell me a joke",
#     "explain machine learning", "who is elon musk", "what is chatgpt",
#     "define cloud computing", "what is blockchain", "what is deep learning", "no of alphabets
# ]

# answers = [
#     "ai means artificial intelligence", "i am fine", "python is a programming language",
#     "why did the computer sleep it had no byte", "machine learning is learning from data",
#     "elon musk is a tech entrepreneur", "chatgpt is an ai chatbot",
#     "cloud computing delivers services over the internet",
#     "blockchain is a distributed ledger", "deep learning uses neural networks", "26", 
# ]

questions = [
    "how does ai work", "how to learn python", "how to create a website", "how to use git",
    "how to debug code", "how does internet work", "how to write functions", "how to use loops",
    "how to handle errors", "how to optimize code", "what is framework", "what is ide",
    "what is syntax", "what is semantics", "what is paradigm", "what is abstraction",
    "how to use arrays", "how to sort data", "how to search algorithms", "how does recursion work",
    "how to manage memory", "how to use pointers", "what is stack", "what is queue",
    "what is tree", "what is graph", "how to implement linked list", "how does hashing work",
    "what is big o notation", "how to analyze complexity", "what is dynamic programming", "what is greedy algorithm",
    "how to design database", "how to normalize data", "what is acid properties", "how to write queries",
    "how to use joins", "how to create indexes", "what is transaction", "how to backup data",
    "how does networking work", "what is osi model", "how to configure router", "what is subnet",
    "how does encryption work", "what is hash function", "how to secure passwords", "what is digital signature",
    "how to prevent sql injection", "what is cross site scripting", "how to implement authentication", "what is session management",
    "how to design ui", "what is design pattern", "how to use mvc", "what is observer pattern",
    "how to implement singleton", "what is factory pattern", "how to use dependency injection", "what is solid principles",
    "how to write unit tests", "what is test driven development", "how to do integration testing", "what is continuous integration",
    "how to deploy applications", "what is blue green deployment", "how to monitor applications", "what is logging",
    "how to handle exceptions", "what is error handling", "how to validate input", "what is sanitization",
    "how to use regular expressions", "what is parsing", "how to work with files", "how to handle csv data",
    "how to connect to api", "what is webhook", "how to implement pagination", "what is rate limiting",
    "how to use async programming", "what is promise", "how to handle callbacks", "what is event loop",
    "how to optimize performance", "what is caching strategy", "how to reduce latency", "what is scalability",
    "how to use version control", "what is branching", "how to merge conflicts", "what is pull request",
    "how to setup environment", "what is virtual environment", "how to manage dependencies", "what is package manager",
    "how to write documentation", "what is code review", "how to refactor code", "what is technical debt",
    "how to learn programming", "what is pseudocode", "how to think algorithmically", "what is problem solving"
]

answers = [
    "ai works by processing data through algorithms and models", "learn python by practicing coding daily and building projects",
    "create website using html css and javascript", "use git by initializing repository and committing changes",
    "debug code by using print statements and debugger tools", "internet works through interconnected networks using protocols",
    "write functions by defining reusable code blocks", "use loops to repeat code execution efficiently",
    "handle errors using try catch blocks", "optimize code by improving algorithms and reducing complexity",
    "framework provides structure and tools for development", "ide is integrated development environment",
    "syntax is the grammar rules of programming language", "semantics is the meaning of code statements",
    "paradigm is programming approach like oop or functional", "abstraction hides complex implementation details",
    "use arrays to store multiple values in single variable", "sort data using algorithms like quicksort or mergesort",
    "search algorithms find specific elements in data structures", "recursion works by function calling itself with base case",
    "manage memory by allocating and deallocating resources properly", "use pointers to reference memory addresses",
    "stack is last in first out data structure", "queue is first in first out data structure",
    "tree is hierarchical data structure with nodes", "graph is collection of vertices connected by edges",
    "implement linked list using nodes with data and pointers", "hashing works by mapping keys to array indexes",
    "big o notation describes algorithm time complexity", "analyze complexity by counting operations relative to input size",
    "dynamic programming solves problems by breaking into subproblems", "greedy algorithm makes locally optimal choices",
    "design database by identifying entities and relationships", "normalize data to eliminate redundancy and anomalies",
    "acid properties ensure database transaction reliability", "write queries using sql select insert update delete",
    "use joins to combine data from multiple tables", "create indexes to improve query performance",
    "transaction is series of database operations as single unit", "backup data regularly using automated scripts",
    "networking works through protocols and layer architecture", "osi model has seven layers for network communication",
    "configure router by setting ip addresses and routing tables", "subnet divides network into smaller segments",
    "encryption works by transforming data using mathematical algorithms", "hash function converts input to fixed size output",
    "secure passwords using strong complexity and hashing", "digital signature verifies authenticity and integrity",
    "prevent sql injection by using parameterized queries", "cross site scripting injects malicious scripts in web pages",
    "implement authentication by verifying user credentials", "session management maintains user state across requests",
    "design ui by focusing on user experience and usability", "design pattern is reusable solution to common problems",
    "use mvc to separate model view and controller concerns", "observer pattern notifies multiple objects of state changes",
    "implement singleton by restricting class to single instance", "factory pattern creates objects without specifying exact class",
    "use dependency injection to provide dependencies externally", "solid principles guide object oriented design",
    "write unit tests to verify individual components work correctly", "test driven development writes tests before implementation",
    "integration testing verifies components work together", "continuous integration automatically builds and tests code",
    "deploy applications using automated deployment pipelines", "blue green deployment reduces downtime during updates",
    "monitor applications using metrics and logging systems", "logging records events and errors for troubleshooting",
    "handle exceptions using structured error handling mechanisms", "error handling manages unexpected conditions gracefully",
    "validate input to ensure data meets expected criteria", "sanitization removes harmful content from user input",
    "use regular expressions to match and manipulate text patterns", "parsing analyzes text according to grammar rules",
    "work with files using read write and append operations", "handle csv data using specialized parsing libraries",
    "connect to api using http requests and handle responses", "webhook sends http callbacks when events occur",
    "implement pagination to display large datasets in chunks", "rate limiting controls number of requests per time period",
    "use async programming to handle concurrent operations efficiently", "promise represents eventual completion of asynchronous operation",
    "handle callbacks by passing functions as parameters", "event loop manages execution of asynchronous operations",
    "optimize performance by profiling and eliminating bottlenecks", "caching strategy stores frequently accessed data in memory",
    "reduce latency by minimizing network delays and processing time", "scalability is ability to handle increased load",
    "use version control to track changes and collaborate", "branching creates separate development lines",
    "merge conflicts occur when same code is modified differently", "pull request proposes changes for review and merging",
    "setup environment by installing required tools and dependencies", "virtual environment isolates project dependencies",
    "manage dependencies using package managers like pip or npm", "package manager installs and updates software packages",
    "write documentation to explain code purpose and usage", "code review examines code for quality and correctness",
    "refactor code to improve structure without changing functionality", "technical debt is cost of poor code quality decisions",
    "learn programming by building projects and solving problems", "pseudocode describes algorithm logic in plain language",
    "think algorithmically by breaking problems into logical steps", "problem solving requires analysis and creative thinking"
]

python_questions = [
    "what is python", "how to install python", "what is python used for", "how to run python code",
    "what are python variables", "how to create python lists", "what are python dictionaries", "how to use python loops",
    "what is python function", "how to import modules in python", "what are python classes", "how to handle exceptions in python",
    "what is python string", "how to read files in python", "what are python tuples", "how to use python sets",
    "what is python comprehension", "how to debug python code", "what are python decorators", "how to use python generators",
    "what is python lambda", "how to connect database in python", "what are python packages", "how to create python virtual environment",
    "what is python pip", "how to parse json in python", "what are python data types", "how to use python regex",
    "what is python oop", "how to create python gui", "what are python modules", "how to test python code",
    "what is python flask", "how to use python django", "what are python libraries", "how to scrape web in python",
    "what is python pandas", "how to use python numpy", "what are python frameworks", "how to deploy python app",
    "what is python threading", "how to use python asyncio", "what are python context managers", "how to profile python code",
    "what is python metaclass", "how to use python iterators", "what are python properties", "how to serialize python objects",
    "what is python closure", "how to use python logging", "what are python annotations", "how to optimize python code",
    "what is python garbage collection", "how to use python multiprocessing", "what are python descriptors", "how to package python code",
    "what is python zen", "how to use python with", "what are python magic methods", "how to handle python memory",
    "what is python scope", "how to use python global", "what are python built-in functions", "how to create python extension",
    "what is python monkey patching", "how to use python slots", "what are python weak references", "how to introspect python objects",
    "what is python mixin", "how to use python super", "what are python abstract classes", "how to implement python interfaces",
    "what is python factory pattern", "how to use python singleton", "what are python design patterns", "how to use python observer",
    "what is python command pattern", "how to use python strategy", "what are python creational patterns", "how to implement python mvc",
    "what is python rest api", "how to use python requests", "what are python web frameworks", "how to create python microservices",
    "what is python celery", "how to use python redis", "what are python message queues", "how to implement python caching",
    "what is python sqlalchemy", "how to use python orm", "what are python database drivers", "how to migrate python database",
    "what is python alembic", "how to use python fixtures", "what are python mocks", "how to test python database",
    "what is python pytest", "how to use python unittest", "what are python test fixtures", "how to measure python coverage",
    "what is python tox", "how to use python continuous integration", "what are python linting tools", "how to format python code",
    "what is python black", "how to use python pylint", "what are python type hints", "how to check python types",
    "what is python mypy", "how to use python pydantic", "what are python data classes", "how to validate python data",
    "what is python enum", "how to use python collections", "what are python default dict", "how to use python counter",
    "what is python named tuple", "how to use python ordered dict", "what are python deque", "how to use python heap",
    "what is python bisect", "how to use python itertools", "what are python functools", "how to use python operator",
    "what is python copy", "how to use python pickle", "what are python json methods", "how to handle python csv",
    "what is python xml parsing", "how to use python beautiful soup", "what are python parsing libraries", "how to handle python yaml",
    "what is python configparser", "how to use python argparse", "what are python command line tools", "how to create python cli",
    "what is python click", "how to use python fire", "what are python automation tools", "how to schedule python tasks",
    "what is python cron", "how to use python scheduler", "what are python monitoring tools", "how to log python applications",
    "what is python elasticsearch", "how to use python kafka", "what are python message brokers", "how to implement python events",
    "what is python docker", "how to use python kubernetes", "what are python deployment tools", "how to containerize python app",
    "what is python fabric", "how to use python ansible", "what are python devops tools", "how to automate python deployment",
    "what is python machine learning", "how to use python scikit learn", "what are python ml libraries", "how to train python models",
    "what is python tensorflow", "how to use python pytorch", "what are python deep learning frameworks", "how to implement python neural networks",
    "what is python keras", "how to use python opencv", "what are python computer vision libraries", "how to process python images",
    "what is python nltk", "how to use python spacy", "what are python nlp libraries", "how to process python text",
    "what is python matplotlib", "how to use python seaborn", "what are python visualization libraries", "how to create python plots",
    "what is python plotly", "how to use python bokeh", "what are python dashboard tools", "how to create python charts",
    "what is python jupyter", "how to use python notebook", "what are python data science tools", "how to analyze python data",
    "what is python statistical analysis", "how to use python scipy", "what are python math libraries", "how to calculate python statistics",
    "what is python sympy", "how to use python numerical computing", "what are python scientific libraries", "how to solve python equations",
    "what is python game development", "how to use python pygame", "what are python game engines", "how to create python games",
    "what is python gui development", "how to use python tkinter", "what are python gui frameworks", "how to create python desktop apps",
    "what is python kivy", "how to use python qt", "what are python cross platform tools", "how to build python mobile apps",
    "what is python security", "how to use python cryptography", "what are python security libraries", "how to encrypt python data",
    "what is python blockchain", "how to use python web3", "what are python cryptocurrency tools", "how to implement python smart contracts",
    "what is python iot", "how to use python raspberry pi", "what are python hardware libraries", "how to control python sensors",
    "what is python automation", "how to use python selenium", "what are python testing frameworks", "how to automate python browser",
    "what is python robotics", "how to use python robotic libraries", "what are python simulation tools", "how to program python robots",
    "what is python networking", "how to use python socket", "what are python network libraries", "how to implement python protocols",
    "what is python concurrency", "how to use python queue", "what are python parallel processing tools", "how to optimize python performance",
    "what is python profiling", "how to use python cProfile", "what are python optimization techniques", "how to benchmark python code",
    "what is python memory management", "how to use python gc", "what are python memory tools", "how to debug python memory leaks",
    "what is python packaging", "how to use python setuptools", "what are python distribution tools", "how to publish python packages",
    "what is python wheel", "how to use python poetry", "what are python dependency managers", "how to manage python versions",
    "what is python pyenv", "how to use python conda", "what are python environment managers", "how to isolate python dependencies",
    "what is python documentation", "how to use python sphinx", "what are python doc tools", "how to generate python docs",
    "what is python code quality", "how to use python flake8", "what are python static analysis tools", "how to maintain python standards",
    "what is python refactoring", "how to use python rope", "what are python code transformation tools", "how to improve python code",
    "what is python migration", "how to use python 2to3", "what are python version upgrade tools", "how to modernize python code",
    "what is python compatibility", "how to use python six", "what are python cross version tools", "how to support python versions",
    "what is python performance", "how to use python cython", "what are python speed optimization tools", "how to accelerate python code",
    "what is python jit", "how to use python numba", "what are python compilation tools", "how to compile python code",
    "what is python alternative implementations", "how to use python pypy", "what are python interpreters", "how to run python faster",
    "what is python embedding", "how to use python c api", "what are python extension tools", "how to integrate python with c",
    "what is python ctypes", "how to use python cffi", "what are python foreign function interfaces", "how to call c from python",
    "what is python swig", "how to use python pybind11", "what are python binding tools", "how to wrap c++ for python",
    "what is python microframeworks", "how to use python fastapi", "what are python async frameworks", "how to build python apis",
    "what is python serverless", "how to use python aws lambda", "what are python cloud functions", "how to deploy python serverless",
    "what is python cloud computing", "how to use python boto3", "what are python cloud libraries", "how to manage python cloud resources",
    "what is python devops", "how to use python terraform", "what are python infrastructure tools", "how to automate python infrastructure",
    "what is python monitoring", "how to use python prometheus", "what are python observability tools", "how to monitor python applications",
    "what is python logging best practices", "how to structure python logs", "what are python log formats", "how to centralize python logging",
    "what is python error tracking", "how to use python sentry", "what are python debugging tools", "how to track python errors",
    "what is python load testing", "how to use python locust", "what are python performance testing tools", "how to test python scalability",
    "what is python api testing", "how to use python requests mock", "what are python testing strategies", "how to test python web services",
    "what is python data validation", "how to use python marshmallow", "what are python schema libraries", "how to validate python json",
    "what is python configuration management", "how to use python hydra", "what are python config tools", "how to manage python settings",
    "what is python feature flags", "how to use python toggle", "what are python feature management tools", "how to control python features",
    "what is python internationalization", "how to use python babel", "what are python i18n tools", "how to localize python apps",
    "what is python accessibility", "how to make python apps accessible", "what are python accessibility tools", "how to test python accessibility",
    "what is python performance monitoring", "how to use python apm", "what are python application monitoring tools", "how to optimize python apps",
    "what is python security scanning", "how to use python bandit", "what are python security tools", "how to secure python applications",
    "what is python dependency scanning", "how to use python safety", "what are python vulnerability tools", "how to check python dependencies",
    "what is python code coverage", "how to use python coverage", "what are python test metrics", "how to measure python test quality",
    "what is python mutation testing", "how to use python mutpy", "what are python test effectiveness tools", "how to improve python tests",
    "what is python property based testing", "how to use python hypothesis", "what are python generative testing tools", "how to test python edge cases",
    "what is python contract testing", "how to use python pact", "what are python integration testing tools", "how to test python service contracts",
    "what is python chaos engineering", "how to use python chaos toolkit", "what are python resilience testing tools", "how to test python system reliability",
    "what is python observability", "how to implement python tracing", "what are python distributed tracing tools", "how to monitor python microservices",
    "what is python metrics collection", "how to use python statsd", "what are python metrics libraries", "how to track python application metrics",
    "what is python alerting", "how to set python alerts", "what are python notification tools", "how to monitor python system health",
    "what is python incident response", "how to handle python outages", "what are python runbook tools", "how to manage python incidents",
    "what is python capacity planning", "how to scale python applications", "what are python scaling strategies", "how to handle python load",
    "what is python disaster recovery", "how to backup python applications", "what are python backup strategies", "how to restore python systems",
    "what is python high availability", "how to make python apps resilient", "what are python redundancy strategies", "how to eliminate python single points of failure",
    "what is python blue green deployment", "how to deploy python without downtime", "what are python deployment strategies", "how to minimize python deployment risk",
    "what is python canary deployment", "how to gradually deploy python changes", "what are python rollout strategies", "how to test python releases safely",
    "what is python feature toggles", "how to control python feature rollout", "what are python release management tools", "how to manage python feature lifecycle"
]

python_answers = [
    "python is a high level interpreted programming language", "install python by downloading from python.org and running installer",
    "python is used for web development data science automation and more", "run python code using python interpreter or ide",
    "python variables store data values using dynamic typing", "create python lists using square brackets and comma separated values",
    "python dictionaries store key value pairs using curly braces", "use python loops like for and while for iteration",
    "python function is reusable block of code defined with def", "import modules using import statement or from import",
    "python classes define objects with attributes and methods", "handle exceptions using try except finally blocks",
    "python string is sequence of characters enclosed in quotes", "read files using open function with read mode",
    "python tuples are immutable sequences defined with parentheses", "use python sets for unique unordered collections",
    "python comprehension creates lists dicts or sets concisely", "debug python code using print statements or debugger",
    "python decorators modify function behavior using @ syntax", "use python generators for memory efficient iteration",
    "python lambda creates anonymous functions for simple operations", "connect database using libraries like sqlite3 or sqlalchemy",
    "python packages are directories containing multiple modules", "create virtual environment using venv or virtualenv",
    "python pip is package installer for managing libraries", "parse json using json.loads and json.dumps methods",
    "python data types include int float str bool list dict", "use python regex with re module for pattern matching",
    "python oop implements classes objects inheritance and polymorphism", "create python gui using tkinter or other frameworks",
    "python modules are files containing python code", "test python code using unittest pytest or other frameworks",
    "python flask is micro web framework for building applications", "use python django for full featured web development",
    "python libraries are collections of modules and packages", "scrape web using beautiful soup or scrapy frameworks",
    "python pandas provides data analysis and manipulation tools", "use python numpy for numerical computing and arrays",
    "python frameworks provide structure for application development", "deploy python app using heroku aws or docker",
    "python threading enables concurrent execution of code", "use python asyncio for asynchronous programming",
    "python context managers handle resource allocation and cleanup", "profile python code using cProfile or other tools",
    "python metaclass controls class creation and behavior", "use python iterators for custom iteration protocols",
    "python properties provide getter setter methods for attributes", "serialize python objects using pickle or json",
    "python closure captures variables from enclosing scope", "use python logging module for application logging",
    "python annotations provide type hints for better code", "optimize python code by improving algorithms and data structures",
    "python garbage collection automatically manages memory", "use python multiprocessing for parallel execution",
    "python descriptors control attribute access in classes", "package python code using setuptools or poetry",
    "python zen is philosophy emphasizing code readability", "use python with statement for context management",
    "python magic methods customize object behavior using dunder methods", "handle python memory using gc module and profiling",
    "python scope determines variable accessibility in code", "use python global keyword for global variables",
    "python built-in functions are always available without import", "create python extension using c api or cython",
    "python monkey patching modifies classes at runtime", "use python slots to optimize memory usage in classes",
    "python weak references allow garbage collection of objects", "introspect python objects using inspect module",
    "python mixin provides functionality to multiple classes", "use python super to call parent class methods",
    "python abstract classes define interface using abc module", "implement python interfaces using abstract base classes",
    "python factory pattern creates objects without specifying class", "use python singleton to restrict class instantiation",
    "python design patterns provide reusable solutions to problems", "use python observer pattern for event handling",
    "python command pattern encapsulates requests as objects", "use python strategy pattern for algorithm selection",
    "python creational patterns deal with object creation", "implement python mvc for separation of concerns",
    "python rest api provides http interface for applications", "use python requests library for http client operations",
    "python web frameworks like flask django facilitate web development", "create python microservices using fastapi or flask",
    "python celery provides distributed task queue functionality", "use python redis for caching and data storage",
    "python message queues enable asynchronous communication", "implement python caching using redis or memcached",
    "python sqlalchemy provides orm for database operations", "use python orm to abstract database interactions",
    "python database drivers connect to specific databases", "migrate python database using alembic or django migrations",
    "python alembic handles database schema migrations", "use python fixtures for test data setup",
    "python mocks simulate objects for testing purposes", "test python database using in memory or test databases",
    "python pytest is modern testing framework with fixtures", "use python unittest for object oriented testing",
    "python test fixtures provide consistent test environment", "measure python coverage using coverage.py tool",
    "python tox tests code across multiple environments", "use python continuous integration with github actions",
    "python linting tools check code quality and style", "format python code using black or autopep8",
    "python black is opinionated code formatter", "use python pylint for static code analysis",
    "python type hints improve code readability and tooling", "check python types using mypy static analyzer",
    "python mypy performs static type checking", "use python pydantic for data validation",
    "python data classes reduce boilerplate for simple classes", "validate python data using schemas and type hints",
    "python enum creates named constants for better code", "use python collections for specialized data structures",
    "python default dict provides default values for missing keys", "use python counter for counting occurrences",
    "python named tuple creates immutable classes with named fields", "use python ordered dict for insertion order preservation",
    "python deque provides efficient append and pop operations", "use python heap for priority queue operations",
    "python bisect maintains sorted lists efficiently", "use python itertools for efficient iteration patterns",
    "python functools provides higher order functions and operations", "use python operator for function versions of operators",
    "python copy module provides shallow and deep copying", "use python pickle for object serialization",
    "python json methods handle json data parsing and generation", "handle python csv using csv module",
    "python xml parsing uses xml.etree or lxml libraries", "use python beautiful soup for html parsing",
    "python parsing libraries handle various data formats", "handle python yaml using pyyaml library",
    "python configparser reads configuration files", "use python argparse for command line argument parsing",
    "python command line tools enhance script usability", "create python cli using click or argparse",
    "python click creates beautiful command line interfaces", "use python fire for automatic cli generation",
    "python automation tools streamline repetitive tasks", "schedule python tasks using cron or apscheduler",
    "python cron schedules tasks using system scheduler", "use python scheduler for time based task execution",
    "python monitoring tools track application performance", "log python applications using logging module",
    "python elasticsearch provides search and analytics", "use python kafka for stream processing",
    "python message brokers handle asynchronous messaging", "implement python events using observer pattern",
    "python docker containerizes applications for deployment", "use python kubernetes for orchestration",
    "python deployment tools automate application deployment", "containerize python app using docker",
    "python fabric automates deployment and administration", "use python ansible for configuration management",
    "python devops tools integrate development and operations", "automate python deployment using ci cd pipelines",
    "python machine learning builds predictive models", "use python scikit learn for machine learning algorithms",
    "python ml libraries provide machine learning functionality", "train python models using supervised learning",
    "python tensorflow builds neural networks and deep learning", "use python pytorch for deep learning research",
    "python deep learning frameworks enable neural networks", "implement python neural networks for pattern recognition",
    "python keras provides high level neural network api", "use python opencv for computer vision tasks",
    "python computer vision libraries process images and video", "process python images using pil or opencv",
    "python nltk provides natural language processing tools", "use python spacy for advanced nlp tasks",
    "python nlp libraries process and analyze text", "process python text using tokenization and analysis",
    "python matplotlib creates static plots and visualizations", "use python seaborn for statistical visualizations",
    "python visualization libraries create charts and graphs", "create python plots for data analysis",
    "python plotly creates interactive web visualizations", "use python bokeh for interactive data visualization",
    "python dashboard tools create web based analytics", "create python charts for business intelligence",
    "python jupyter provides interactive computing environment", "use python notebook for data science workflows",
    "python data science tools enable data analysis", "analyze python data using pandas and numpy",
    "python statistical analysis performs statistical tests", "use python scipy for scientific computing",
    "python math libraries provide mathematical functions", "calculate python statistics using scipy.stats",
    "python sympy performs symbolic mathematics", "use python numerical computing for scientific applications",
    "python scientific libraries enable scientific computing", "solve python equations using numerical methods",
    "python game development creates interactive games", "use python pygame for 2d game development",
    "python game engines provide game development frameworks", "create python games using pygame or panda3d",
    "python gui development creates desktop applications", "use python tkinter for cross platform guis",
    "python gui frameworks provide user interface tools", "create python desktop apps using gui frameworks",
    "python kivy creates cross platform applications", "use python qt for professional desktop applications",
    "python cross platform tools work on multiple operating systems", "build python mobile apps using kivy or beeware",
    "python security protects applications from threats", "use python cryptography for encryption and security",
    "python security libraries provide security functionality", "encrypt python data using cryptographic algorithms",
    "python blockchain builds decentralized applications", "use python web3 for blockchain interaction",
    "python cryptocurrency tools interact with crypto networks", "implement python smart contracts using web3",
    "python iot connects physical devices to internet", "use python raspberry pi for iot projects",
    "python hardware libraries control physical devices", "control python sensors using gpio libraries",
    "python automation automates repetitive tasks", "use python selenium for web browser automation",
    "python testing frameworks provide test automation", "automate python browser testing using selenium",
    "python robotics controls robotic systems", "use python robotic libraries for robot programming",
    "python simulation tools model real world systems", "program python robots using robotic frameworks",
    "python networking enables network communication", "use python socket for network programming",
    "python network libraries provide networking functionality", "implement python protocols for communication",
    "python concurrency enables parallel execution", "use python queue for thread safe communication",
    "python parallel processing tools optimize performance", "optimize python performance using concurrency",
    "python profiling identifies performance bottlenecks", "use python cProfile for performance analysis",
    "python optimization techniques improve code efficiency", "benchmark python code for performance measurement",
    "python memory management handles memory allocation", "use python gc for garbage collection control",
    "python memory tools analyze memory usage", "debug python memory leaks using memory profilers",
    "python packaging distributes code as packages", "use python setuptools for package creation",
    "python distribution tools publish packages", "publish python packages to pypi repository",
    "python wheel is binary distribution format", "use python poetry for dependency management",
    "python dependency managers handle package dependencies", "manage python versions using version managers",
    "python pyenv manages multiple python versions", "use python conda for package and environment management",
    "python environment managers isolate project dependencies", "isolate python dependencies using virtual environments",
    "python documentation explains code functionality", "use python sphinx for documentation generation",
    "python doc tools create and publish documentation", "generate python docs from docstrings",
    "python code quality ensures maintainable code", "use python flake8 for code quality checking",
    "python static analysis tools analyze code without execution", "maintain python standards using style guides",
    "python refactoring improves code structure", "use python rope for automated refactoring",
    "python code transformation tools modify code structure", "improve python code through refactoring",
    "python migration updates code to newer versions", "use python 2to3 for python version migration",
    "python version upgrade tools modernize code", "modernize python code for current standards",
    "python compatibility ensures code works across versions", "use python six for python 2 3 compatibility",
    "python cross version tools support multiple python versions", "support python versions using compatibility libraries",
    "python performance optimizes code execution speed", "use python cython for performance optimization",
    "python speed optimization tools accelerate code", "accelerate python code using compiled extensions",
    "python jit compilation improves runtime performance", "use python numba for just in time compilation",
    "python compilation tools convert python to machine code", "compile python code for better performance",
    "python alternative implementations provide different interpreters", "use python pypy for faster execution",
    "python interpreters execute python code", "run python faster using optimized interpreters",
    "python embedding integrates python into other applications", "use python c api for embedding python",
    "python extension tools create native extensions", "integrate python with c using extension modules",
    "python ctypes calls c libraries from python", "use python cffi for foreign function interface",
    "python foreign function interfaces call external libraries", "call c from python using ctypes",
    "python swig generates bindings for c c++", "use python pybind11 for c++ binding",
    "python binding tools wrap native libraries", "wrap c++ for python using binding generators",
    "python microframeworks provide minimal web frameworks", "use python fastapi for modern api development",
    "python async frameworks support asynchronous web development", "build python apis using async frameworks",
    "python serverless runs code without server management", "use python aws lambda for serverless computing",
    "python cloud functions execute code in cloud", "deploy python serverless using cloud platforms",
    "python cloud computing utilizes cloud resources", "use python boto3 for aws integration",
    "python cloud libraries interact with cloud services", "manage python cloud resources using sdks",
    "python devops integrates development and operations", "use python terraform for infrastructure as code",
    "python infrastructure tools manage cloud infrastructure", "automate python infrastructure using devops tools",
    "python monitoring tracks application performance", "use python prometheus for metrics collection",
    "python observability tools provide system visibility", "monitor python applications using observability platforms",
    "python logging best practices ensure effective logging", "structure python logs for better analysis",
    "python log formats standardize log output", "centralize python logging using log aggregation",
    "python error tracking identifies and fixes bugs", "use python sentry for error monitoring",
    "python debugging tools help find and fix issues", "track python errors using error monitoring services",
    "python load testing evaluates system performance under load", "use python locust for load testing",
    "python performance testing tools measure system performance", "test python scalability using performance tools",
    "python api testing validates api functionality", "use python requests mock for api testing",
    "python testing strategies ensure code quality", "test python web services using testing frameworks",
    "python data validation ensures data integrity", "use python marshmallow for data serialization",
    "python schema libraries define data structures", "validate python json using schema validation",
    "python configuration management handles application settings", "use python hydra for configuration management",
    "python config tools manage application configuration", "manage python settings using configuration libraries",
    "python feature flags control feature availability", "use python toggle for feature flag management",
    "python feature management tools control feature rollout", "control python features using feature flags",
    "python internationalization supports multiple languages", "use python babel for internationalization",
    "python i18n tools handle localization", "localize python apps for global audiences",
    "python accessibility ensures apps work for all users", "make python apps accessible using accessibility guidelines",
    "python accessibility tools test accessibility compliance", "test python accessibility using automated tools",
    "python performance monitoring tracks application metrics", "use python apm for application performance monitoring",
    "python application monitoring tools provide insights", "optimize python apps using performance monitoring",
    "python security scanning identifies vulnerabilities", "use python bandit for security analysis",
    "python security tools protect applications", "secure python applications using security best practices",
    "python dependency scanning checks for vulnerabilities", "use python safety for dependency vulnerability checking",
    "python vulnerability tools identify security issues", "check python dependencies for known vulnerabilities",
    "python code coverage measures test effectiveness", "use python coverage for test coverage analysis",
    "python test metrics evaluate test quality", "measure python test quality using coverage metrics",
    "python mutation testing evaluates test effectiveness", "use python mutpy for mutation testing",
    "python test effectiveness tools improve test quality", "improve python tests using mutation testing",
    "python property based testing generates test cases", "use python hypothesis for property based testing",
    "python generative testing tools create comprehensive tests", "test python edge cases using property based testing",
    "python contract testing validates service interactions", "use python pact for contract testing",
    "python integration testing tools test service integration", "test python service contracts using contract testing",
    "python chaos engineering tests system resilience", "use python chaos toolkit for chaos engineering",
    "python resilience testing tools test system reliability", "test python system reliability using chaos engineering",
    "python observability provides system visibility", "implement python tracing for request tracking",
    "python distributed tracing tools track requests", "monitor python microservices using distributed tracing",
    "python metrics collection gathers performance data", "use python statsd for metrics collection",
    "python metrics libraries track application metrics", "track python application metrics using metrics libraries",
    "python alerting notifies of system issues", "set python alerts for system monitoring",
    "python notification tools send alerts", "monitor python system health using alerting",
    "python incident response handles system outages", "handle python outages using incident response procedures",
    "python runbook tools automate incident response", "manage python incidents using runbook automation",
    "python capacity planning ensures adequate resources", "scale python applications for increased load",
    "python scaling strategies handle growth", "handle python load using scaling strategies",
    "python disaster recovery ensures business continuity", "backup python applications for disaster recovery",
    "python backup strategies protect against data loss", "restore python systems from backups",
    "python high availability ensures system uptime", "make python apps resilient using redundancy",
    "python redundancy strategies eliminate single points of failure", "eliminate python single points of failure using redundancy",
    "python blue green deployment enables zero downtime deployments", "deploy python without downtime using blue green deployment",
    "python deployment strategies minimize deployment risk", "minimize python deployment risk using deployment strategies",
    "python canary deployment gradually rolls out changes", "gradually deploy python changes using canary deployment",
    "python rollout strategies control release pace", "test python releases safely using rollout strategies",
    "python feature toggles control feature availability", "control python feature rollout using feature toggles",
    "python release management tools manage software releases", "manage python feature lifecycle using release management"
]


data = [f"user: {q} assistant: {a}" for q,a in zip(python_questions, python_answers)] * 100  # 1000 samples


In [75]:
# Build a simple char-level tokenizer
chars = sorted(list(set("".join(data))))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s): return [stoi[c] for c in s]
def decode(l): return "".join([itos[i] for i in l])


In [76]:
block_size = 64  # context length

def get_batch(batch_size=16):
    inputs, targets = [], []
    for _ in range(batch_size):
        text = random.choice(data)
        ix = encode(text)
        # ✨ Ensure every sequence is exactly block_size+1
        if len(ix) < block_size + 1:
            ix = ix + [0] * (block_size + 1 - len(ix))  # pad with 0s
        else:
            ix = ix[:block_size + 1]  # truncate if too long
        x = torch.tensor(ix[:-1], dtype=torch.long)
        y = torch.tensor(ix[1:], dtype=torch.long)
        inputs.append(x)
        targets.append(y)
    return torch.stack(inputs).to(device), torch.stack(targets).to(device)


In [77]:
# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [78]:
# Self-Attention
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.heads = heads
        self.head_dim = embed_dim // heads

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x).reshape(B, T, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        tril = torch.tril(torch.ones(T, T, device=x.device))
        scores = scores.masked_fill(tril == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().reshape(B, T, C)
        return self.fc_out(out)


In [79]:
# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, heads, ff_hidden_dim):
        super().__init__()
        self.attn = SelfAttention(embed_dim, heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_hidden_dim),
            nn.ReLU(),
            nn.Linear(ff_hidden_dim, embed_dim)
        )

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x


In [80]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, heads=4, layers=2, ff_hidden_dim=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, heads, ff_hidden_dim) for _ in range(layers)])
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, targets=None):
        x = self.embed(x)
        x = self.pos_encoding(x)
        x = self.blocks(x)
        logits = self.fc_out(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    def generate(self, idx, max_new_tokens=50):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx


In [81]:
model = MiniGPT(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(2000):  # small training
    x, y = get_batch()
    logits, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")


Step 0 | Loss: 3.6823
Step 50 | Loss: 1.8199
Step 100 | Loss: 1.4457
Step 150 | Loss: 1.1936
Step 200 | Loss: 1.0996
Step 250 | Loss: 1.1421
Step 300 | Loss: 1.0251
Step 350 | Loss: 1.0336
Step 400 | Loss: 0.8998
Step 450 | Loss: 0.9536
Step 500 | Loss: 0.8260
Step 550 | Loss: 0.8175
Step 600 | Loss: 0.7866
Step 650 | Loss: 0.8593
Step 700 | Loss: 0.8625
Step 750 | Loss: 0.7621
Step 800 | Loss: 0.9001
Step 850 | Loss: 0.8033
Step 900 | Loss: 0.7012
Step 950 | Loss: 0.7560
Step 1000 | Loss: 0.7487
Step 1050 | Loss: 0.7269
Step 1100 | Loss: 0.6570
Step 1150 | Loss: 0.6509
Step 1200 | Loss: 0.6292
Step 1250 | Loss: 0.5636
Step 1300 | Loss: 0.5960
Step 1350 | Loss: 0.6088
Step 1400 | Loss: 0.5382
Step 1450 | Loss: 0.5796
Step 1500 | Loss: 0.5904
Step 1550 | Loss: 0.5151
Step 1600 | Loss: 0.5000
Step 1650 | Loss: 0.5440
Step 1700 | Loss: 0.5357
Step 1750 | Loss: 0.5602
Step 1800 | Loss: 0.3897
Step 1850 | Loss: 0.4763
Step 1900 | Loss: 0.4044
Step 1950 | Loss: 0.4195


In [124]:
context = torch.tensor([encode("user: what is ai assistant:")], dtype=torch.long).to(device)
output = model.generate(context, max_new_tokens=50)
print("Output:\n", decode(output[0].tolist()))


Output:
 user: what is ai assistant: python aotes assist asistant: chalt park ons libr


In [125]:
context = torch.tensor([encode("user : what is python lambda :")], dtype=torch.long).to(device)
output = model.generate(context, max_new_tokens=50)
print("Output:\n", decode(output[0].tolist()))


Output:
 user : what is python lambda :faka python iseneck ommblad int: packinoncracsorar


In [46]:
# # pip install wikipedia-api pandas requests tqdm
# !pip install wikipedia

In [44]:
# import pandas as pd
# import wikipedia

# # Set up Wikipedia API with language and user agent
# wikipedia.set_lang("en")
# wikipedia.set_user_agent("mini-chatgpt-app/1.0 (contact@example.com)")

# # Define your questions
# questions = [
#     "What is Artificial Intelligence?",
#     "What is Machine Learning?",
#     "Who is Elon Musk?",
#     "What is Python programming language?",
#     "What is Deep Learning?",
#     "What is Blockchain?",
#     "What is Cloud Computing?",
#     "What is Quantum Computing?",
#     "What is Neural Network?",
#     "What is ChatGPT?"
# ]

# data = []

# for q in questions:
#     try:
#         # Get the summary from Wikipedia
#         summary = wikipedia.summary(q, sentences=3)  # limit to 3 sentences
#         data.append({"question": q, "answer": summary})
#         print(f"✅ Retrieved: {q}")
#     except Exception as e:
#         print(f"⚠️ Skipped {q} — {e}")


In [45]:
# df = pd.DataFrame(data)
# df.to_csv("wiki_qa_dataset.csv", index=False)
# print("✅ Saved to wiki_qa_dataset.csv")
# df.head()
